# 11 · 规则挖掘模块（Rule Mining）功能演示

演示单特征、多特征、多标签规则挖掘与决策树规则提取，以及基于 Rule 的规则评估指标。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 单特征规则挖掘 SingleFeatureRuleMiner
对每个特征自动分箱并挖掘高 Lift 的单条件规则。

In [2]:
from hscredit.report.mining import (SingleFeatureRuleMiner, MultiFeatureRuleMiner,
    MultiLabelRuleMiner, TreeRuleExtractor, RuleMetrics, calculate_rule_metrics)

dft = df.copy(); dft['target'] = y.values; dft['label2'] = (dft['CURRENT_DPD'] > 7).astype(int)
X = dft[NUM_FEATURES + ['target']]

single = SingleFeatureRuleMiner(target='target', min_lift=1.2, min_samples=10)
single.fit(X)
single_rules = single.get_rules()
single_rules if isinstance(single_rules, pd.DataFrame) else pd.DataFrame(single_rules)

,0
0,Rule('衡枢鉴真分老客版 >= 0.1157')
1,Rule('近六个月非银多头机构数 >= 72.0')
2,Rule('青云24 <= 552.0')
3,Rule('青云24 <= 608.0')
4,Rule('衡枢鉴真分老客版 >= 0.0719')


## 2. 多特征组合规则挖掘 MultiFeatureRuleMiner

In [3]:
multi = MultiFeatureRuleMiner(target='target', min_lift=1.1, max_n_bins=4)
multi.fit(X)
mr = multi.get_rules()
(mr if isinstance(mr, pd.DataFrame) else pd.DataFrame(mr)).head(10)

,0
0,"Rule(""(珊瑚92 == '[-inf, 589.5)') & (青云24 == '[603.00, 647.00)')"")"
1,"Rule(""(衡枢鉴真分老客版 == '[0.0541, 0.0838)') & (近六个月非银多头机构数 == '[69, +inf)')"")"
2,"Rule(""(衡枢鉴真分老客版 == '[0.1239, +inf)') & (近六个月非银多头机构数 == '[69, +inf)')"")"
3,"Rule(""(衡枢鉴真分老客版 == '[0.1239, +inf)') & (天创小额网贷分 == '[747.0, +inf)')"")"
4,"Rule(""(衡枢鉴真分老客版 == '[0.0541, 0.0838)') & (天创小额网贷分 == '[-inf, 674.5)')"")"
5,"Rule(""(衡枢鉴真分老客版 == '[0.0541, 0.0838)') & (天创小额网贷分 == '[747.0, +inf)')"")"
6,"Rule(""(衡枢鉴真分老客版 == '[0.1239, +inf)') & (天创小额网贷分 == '[-inf, 674.5)')"")"
7,"Rule(""(衡枢鉴真分老客版 == '[0.1239, +inf)') & (天创小额网贷分 == '[674.5, 712.0)')"")"
8,"Rule(""(衡枢鉴真分老客版 == '[0.1239, +inf)') & (天创小额网贷分 == '[712.0, 747.0)')"")"
9,"Rule(""(衡枢鉴真分老客版 == '[0.0838, 0.1239)') & (占信V3 == '[-inf, 534)')"")"


## 3. 多标签规则挖掘 MultiLabelRuleMiner（同时评估多个逾期标签）

In [4]:
ml = MultiLabelRuleMiner(labels=['target','label2'], min_lift=1.2, n_bins=6)
ml.fit(dft[NUM_FEATURES + ['target','label2']])
ml_rules = ml.get_rules()
(ml_rules if isinstance(ml_rules, pd.DataFrame) else pd.DataFrame(ml_rules)).head(10)

,规则,覆盖样本数,覆盖率,target_坏率,target_LIFT,target_有效,label2_坏率,label2_LIFT,label2_有效,规则类型,建议
0,衡枢鉴真分老客版 >= 0.1157,291,30.0000,21.3100,1.5196,True,33.6800,1.2421,True,强规则（全标签有效）,稳定拒绝规则
1,青云24 <= 608.0,513,52.8900,17.1500,1.2235,True,32.9400,1.2150,True,强规则（全标签有效）,稳定拒绝规则
2,天创小额网贷分 <= 693.0,363,37.4200,17.0800,1.2182,True,32.7800,1.2091,True,强规则（全标签有效）,稳定拒绝规则
3,占信V3 <= 578.0,489,50.4100,16.9700,1.2106,True,31.7000,1.1691,False,部分有效（target）,谨慎使用/预警规则


## 4. 决策树规则提取 TreeRuleExtractor（决策树 / 随机森林）

In [5]:
tree = TreeRuleExtractor(target='target', max_depth=4, algorithm='dt')
tree.fit(X)
tree_rules = tree.get_rules_dataframe()
tree_rules.head(10)

,规则编号,规则表达式,命中样本数,命中样本占比,命中坏样本率,命中LIFT值,坏账改善
0,3,(衡枢鉴真分老客版 <= 0.15583351254463196) & (占信V3 > 492.5) & (近六个月非银多头机构数 <= 85.5) & (占信V3 <= 608.5),None,None,None,None,None
1,8,(衡枢鉴真分老客版 > 0.15583351254463196) & (衡枢鉴真分老客版 <= 0.2080056369304657) & (占信V3 > 530.5) & (近六个月非银多头机构数 <= 78.5),None,None,None,None,None
2,1,(衡枢鉴真分老客版 <= 0.15583351254463196) & (占信V3 <= 492.5) & (近六个月非银多头机构数 <= 74.5) & (衡枢鉴真分老客版 > 0.05450434610247612),None,None,None,None,None
3,4,(衡枢鉴真分老客版 <= 0.15583351254463196) & (占信V3 > 492.5) & (近六个月非银多头机构数 <= 85.5) & (占信V3 > 608.5),None,None,None,None,None
4,7,(衡枢鉴真分老客版 > 0.15583351254463196) & (衡枢鉴真分老客版 <= 0.2080056369304657) & (占信V3 <= 530.5) & (衡枢鉴真分老客版 > 0.16863519698381424),None,None,None,None,None
5,0,(衡枢鉴真分老客版 <= 0.15583351254463196) & (占信V3 <= 492.5) & (近六个月非银多头机构数 <= 74.5) & (衡枢鉴真分老客版 <= 0.05450434610247612),None,None,None,None,None


## 5. 规则评估 RuleMetrics（基于 Rule 对象计算训练/测试指标）

In [6]:
from hscredit.core.rules import Rule
rm = RuleMetrics()
rule = Rule("`衡枢鉴真分老客版` < 600", name="低分")
metrics = calculate_rule_metrics(rule, dft[NUM_FEATURES], y)
pd.Series(metrics)

训练_命中样本数     970.0000
训练_命中样本占比      1.0000
训练_命中坏样本数    136.0000
训练_命中好样本数    834.0000
训练_命中坏样本率      0.1402
训练_命中LIFT值     1.0000
训练_准确率         0.1402
训练_精确率         0.1402
训练_召回率         1.0000
训练_F1值         0.2459
训练_真正例       136.0000
训练_假正例       834.0000
训练_真负例         0.0000
训练_假负例         0.0000
训练_拦截后坏样本率     0.0000
训练_坏账改善        1.0000
dtype: float64

## 6. 挖掘结果导出 Excel

In [7]:
out_rules = single_rules if isinstance(single_rules, pd.DataFrame) else pd.DataFrame(single_rules)
out_rules.to_excel(f"{OUT}/11_mining_single_rules.xlsx", index=False)
tree_rules.to_excel(f"{OUT}/11_mining_tree_rules.xlsx", index=False)
print('已保存规则挖掘结果')

已保存规则挖掘结果
